# Static SHRED for NOAA data (using my methods)

Dataset: Weekly mean sea-surface temperature as given by the NOAA Optimum Interpolation SST V2 dataset [[1]](https://psl.noaa.gov/data/gridded/data.noaa.oisst.v2.html)

Model: Static SHRED from [[2]](https://royalsocietypublishing.org/rspa/article/480/2298/20240054/66770/Sensing-with-shallow-recurrent-decoder) [[3]](https://github.com/Jan-Williams/pyshred/tree/main)

In [ ]:
# Standard imports
import json
import os
import random

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import ListedColormap
from sklearn.preprocessing import MinMaxScaler

# Local imports
import models  # https://github.com/Jan-Williams/pyshred/blob/main/models.py 
from Data.preprocess_sst_ofam3_shred import (
    convert_to_anomaly,
    create_shred_sequences,
    split_ordered_data,
)
from helper_functions.evaluation import area_weighted_mean, evaluate_metrics
from helper_functions.noaa_shred import load_data
from helper_functions.plotting_functions import plot_compare_sst_recon, plot_sst_map
from helper_functions.saving import JsonEncoder, unpack_metadata

np.random.seed(42)
random.seed(42)

## Load data
From Mobile-Sensor SHRED paper [[4]](https://ieeexplore.ieee.org/document/10584544):

- Weekly mean SST from 1992 - 2019
- 1400 snapshots of a 180 x 360 grid, with landmass excluded leaves 44,219 spatial locations corresponding to the sea surface

In [ ]:
data, lats, longs, dates = load_data()
print(data.shape, len(longs), len(lats))

In [ ]:
df = pd.DataFrame({"Longitude": longs, "Latitude": lats, "SST": data[0]})
plot_sst_map(df)

In [ ]:
exp_name = "NOAA/Exp1"
num_sensors = 3
lags = 52
hidden_dim = 64
sequence_model = "LSTM"
anomaly = False

## SHRED preprocessing - STOP HERE IF ALREADY TRAINED

In [ ]:
print("Preprocessing data...")
train_data, val_data, test_data, train_dates, val_dates, test_dates = split_ordered_data(data, dates, all_dates=True)
if anomaly:
    train_anomaly, val_anomaly, test_anomaly, bulk_mean = convert_to_anomaly((train_data, val_data, test_data))

In [ ]:
print(train_data.shape)
print(val_data.shape)
print(test_data.shape)
print(len(train_data)+len(test_data)+len(val_data))

In [ ]:
print(train_dates.shape)
print(val_dates.shape)
print(test_dates.shape)

In [ ]:
df = pd.DataFrame({"Longitude": longs, "Latitude": lats, "SST": data[1]})
plot_sst_map(df)

In [ ]:
# Plots for santity checks:
plot_index = 0
sst_df = pd.DataFrame({
    "Longitude": longs,
    "Latitude": lats,
    "SST": train_data[plot_index]
})
plot_sst_map(sst_df, area="NA", cmap="plasma", title=f"Actual SST, Date: {train_dates[plot_index]}")

if anomaly:
    anomaly_df = pd.DataFrame({
        "Longitude": longs,
        "Latitude": lats,
        "SST": train_anomaly[plot_index]
    })
    plot_sst_map(anomaly_df, area="NA", cmap="viridis", title=f"SST Anomaly, Date: {train_dates[plot_index]}")

    bulk_df = pd.DataFrame({
        "Longitude": longs,
        "Latitude": lats,
        "SST": bulk_mean
    })
    plot_sst_map(bulk_df, area="NA", cmap="plasma", title=f"Bulk mean SST, Dates: {train_dates[0]} to {train_dates[-1]}")

In [ ]:
# Normalise data
if anomaly:
    sc = MinMaxScaler()
    sc = sc.fit(train_anomaly)  # Computes min and max from training data only (prevent data leakage)
    train_transformed = sc.transform(train_anomaly)  
    val_transformed = sc.transform(val_anomaly)
    test_transformed = sc.transform(test_anomaly)
else:
    sc = MinMaxScaler()
    sc = sc.fit(train_data)  # Computes min and max from training data only (prevent data leakage)
    train_transformed = sc.transform(train_data)  
    val_transformed = sc.transform(val_data)
    test_transformed = sc.transform(test_data)

# Build sequences
sensor_locations = np.random.choice(data.shape[1], size=num_sensors, replace=False)
sensor_coords=(longs[sensor_locations], lats[sensor_locations])
print(f"Sensor location indexes will be: {sensor_locations}")

# Build sequences
train_dataset = create_shred_sequences(train_transformed, sensor_locations, lags=lags)
val_dataset = create_shred_sequences(val_transformed, sensor_locations, lags=lags)
test_dataset = create_shred_sequences(test_transformed, sensor_locations, lags=lags)

# Get dates corresponding to the predictions made by the model
train_lag_dates = train_dates[np.array(range(len(train_dates)-lags)) + lags - 1]
val_lag_dates = val_dates[np.array(range(len(val_dates)-lags)) + lags - 1]
test_lag_dates = test_dates[np.array(range(len(test_dates)-lags)) + lags - 1]

In [ ]:
print(train_lag_dates.shape)
print(val_lag_dates.shape)
print(test_lag_dates.shape)

In [ ]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))
print(len(train_dataset) + len(val_dataset) + len(test_dataset))

In [ ]:
plot_index = 3
plot_data = test_dataset[plot_index]
random_colours = ["#"+''.join([random.choice('0123456789ABCDEF') for j in range(6)])
            for i in range(num_sensors)]

for i in range(num_sensors):
    # plot_data[0] -> test sequences
    # plot_data[0].T -> for each sensor, timeseries of SST values across test dataset
    plt.plot(plot_data[0].T[i], c=random_colours[i])
plt.ylabel("SST")
plt.xlabel("Week index")
plt.grid()
plt.xlim(0, 52)
plt.show()

## Train + test SHRED model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
shred = models.SHRED(num_sensors, len(lats), seq_model=sequence_model,
                     hidden_size=hidden_dim, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)
validation_errors = models.fit(shred, train_dataset, val_dataset, batch_size=64, num_epochs=1000, lr=1e-3, verbose=True, patience=5)

In [ ]:
test_recons = sc.inverse_transform(shred(test_dataset.X).detach().cpu().numpy())
test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())
print('Test Reconstruction Error: ')
print(np.linalg.norm(test_recons - test_ground_truth) / np.linalg.norm(test_ground_truth))

## Save results

In [ ]:
# Make experiment folder
directory = f"Reconstructions/Static/{exp_name}/"
os.makedirs(directory, exist_ok = True)

In [ ]:
np.save(f"Reconstructions/Static/{exp_name}/recons_n{num_sensors}_l{lags}", test_recons)
np.save(f"Reconstructions/Static/{exp_name}/truth_n{num_sensors}_l{lags}", test_ground_truth)

In [ ]:
metadata = {
    "sensor_locations": list(sensor_locations),
    "sensor_coords": list(sensor_coords),
    "test_lag_dates": list(test_lag_dates)
    }

if anomaly:
    metadata["bulk_mean"] = bulk_mean

In [ ]:
with open(f"Reconstructions/Static/{exp_name}/metadata_n{num_sensors}_l{lags}.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, cls=JsonEncoder)

# Evaluation - START HERE if already trained

## Original SHRED pipeline vs my pipeline with data leakage fixed

In [ ]:
colours = [ "#ffb000", "#dc267f", "#1589e8", "#631ff3", "#fe5100", "#000000"]  # IBM colourblind palette  green: "#0d894b"

In [ ]:
test_recons_mine = np.load(f"Reconstructions/Static/{exp_name}/recons_n{num_sensors}_l{lags}.npy")
ground_truth_mine = np.load(f"Reconstructions/Static/{exp_name}/truth_n{num_sensors}_l{lags}.npy")
sensor_locations, sensor_coords, _ = unpack_metadata(f"Reconstructions/Static/{exp_name}/metadata_n{num_sensors}_l{lags}.json")

orig_name = "NOAA/OriginalSHRED"
test_recons_original = np.load(f"Reconstructions/Static/{orig_name}/recons_n{num_sensors}_l{lags}.npy")
ground_truth_original = np.load(f"Reconstructions/Static/{orig_name}/truth_n{num_sensors}_l{lags}.npy")
sensor_locs_original = unpack_metadata(f"Reconstructions/Static/{orig_name}/metadata_n{num_sensors}_l{lags}.json")
assert np.unique(sensor_locs_original == sensor_locations)[0]

In [ ]:
lag_indices_mine = np.arange(len(test_recons_mine))
lag_indices_original = np.arange(len(test_recons_original))

In [ ]:
print("SHRED, my pipeline\nNUM SENSORS: ", num_sensors, "LAGS: ", lags)
evaluate_metrics(test_recons_mine, ground_truth_mine)

print("SHRED, original pipeline\nNUM SENSORS: ", num_sensors, "LAGS: ", lags)
evaluate_metrics(test_recons_original, ground_truth_original)

In [ ]:
difference = test_recons_mine - ground_truth_mine
plot_indexes = [3]
for i in plot_indexes:
    diff_df = pd.DataFrame({
                "Latitude": lats,
                "Longitude": longs,
                "SST": difference[i]
            })
    recon_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons[i]
    })
    true_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_ground_truth[i]
    })
    # plot_sst_map(diff_df, title=f"Difference map for {test_dates[i]}", min_sst=-5, max_sst=5, area="NA", cmap='bwr')
    print(f"My Pipeline Reconstruction: Lag index: {lag_indices[i]}")
    plot_compare_sst_recon(recon_df, true_df, diff_data=diff_df)  #, sensor_coords=sensor_coords)

### Regional integral of SST (area weighted mean)

In [ ]:
for test_recons, ground_truth, lag_indices in [(test_recons_mine, ground_truth_mine, lag_indices_mine)]:
    print("MY PIPELINE")
    truth_sst_mean, _ = area_weighted_mean(ground_truth, lats, longs)
    test_sst_mean, _ = area_weighted_mean(test_recons, lats, longs)

    sst_mean_df = pd.DataFrame({
        "Lag index": lag_indices,
        f"SHRED (LSTM) - {num_sensors} sensors": test_sst_mean,
        "Truth": truth_sst_mean
    })
    sst_mean_df = sst_mean_df.set_index("Lag index")

    plt.figure(figsize=(8,8))
    
    sst_mean_df[[f"SHRED (LSTM) - {num_sensors} sensors",
                "Truth"]].plot(colormap=ListedColormap(colours[:2]), ls='--')
    plt.xticks(rotation=90)
    plt.ylabel("Mean SST ($\\degree$C)")
    plt.legend(bbox_to_anchor=(1,1))
    plt.show()

In [ ]:
# PROBLEM: this doesn't really show much :( consequence of shuffled test set

truth_sst_mean, _ = area_weighted_mean(ground_truth_original, lats, longs)
test_sst_mean, _ = area_weighted_mean(test_recons_original, lats, longs)

sst_mean_df = pd.DataFrame({
    f"SHRED (LSTM) - {num_sensors} sensors": test_sst_mean,
    "Truth": truth_sst_mean
})

plt.figure(figsize=(8,8))
plt.scatter(lag_indices_original, sst_mean_df[f"SHRED (LSTM) - {num_sensors} sensors"], color=colours[0], label=f"SHRED (LSTM) - {num_sensors} sensors")
plt.scatter(lag_indices_original, sst_mean_df["Truth"], color=colours[1], label="Truth")
plt.xticks(rotation=90)
plt.ylabel("Mean SST ($\\degree$C)")
plt.legend(bbox_to_anchor=(1,1))
plt.show()

### Difference
#### Area-weighted average difference

In [ ]:
difference_mine = test_recons_mine - ground_truth_mine
mean_diff_mine, _ = area_weighted_mean(difference_mine, lats, longs)
difference_original = test_recons_original - ground_truth_original
mean_diff_original, _ = area_weighted_mean(difference_original, lats, longs)

In [ ]:
for mean_diff, lag_indices, plot_type in [(mean_diff_mine, lag_indices_mine, "line"),
                                  (mean_diff_original, lag_indices_original, "scatter")]:
    sst_diff_df = pd.DataFrame({
        "Lag index": lag_indices,
        f"SHRED (LSTM) - {num_sensors} sensors": mean_diff
    })

    sst_diff_df = sst_diff_df.set_index("Lag index")

    plt.figure(figsize=(8,8))
    if plot_type == "line":
        sst_diff_df[[f"SHRED (LSTM) - {num_sensors} sensors"]].plot(colormap=ListedColormap(colours[:5]), ls='-')
    elif plot_type == "scatter": 
        plt.scatter(lag_indices_original, sst_diff_df[f"SHRED (LSTM) - {num_sensors} sensors"], color=colours[0], label=f"SHRED (LSTM) - {num_sensors} sensors")
    plt.xticks(rotation=90)
    plt.hlines(0, lag_indices[0], lag_indices[-1], colors=['k'], linestyles=['--'])
    plt.xlim(lag_indices[0], lag_indices[-1])
    plt.ylabel("Average difference (from the truth) ($\\degree$C)")
    plt.grid()
    plt.legend(bbox_to_anchor=(1,1))
    plt.show()

#### Zonally integrated average difference

In [ ]:
avg_all_time_lat_avg_diffs = []
labels = []
for (num_sensors, model_str, difference) in [(num_sensors, "SHRED (LSTM) - my pipeline", difference_mine),
                                             (num_sensors, "SHRED (LSTM) - original", difference_original)]:
    
    # For each reconstruction at a set date, find the average difference at each latitude
    latitude_avg_diffs = np.array([pd.Series(diff_at_time).groupby(lats).mean().to_numpy() for diff_at_time in difference])

    # For each latitude, calculate the mean difference over all time
    avg_all_time_lat_avg_diffs.append(np.array([np.mean(latitude_avg_diffs.T[j]) for j in range(latitude_avg_diffs.shape[1])]))
    labels.append(f"{model_str} - {num_sensors} sensors")

In [ ]:
colour_index = 0
for i in range(len(avg_all_time_lat_avg_diffs)):
    plt.plot(avg_all_time_lat_avg_diffs[i], np.unique(lats), c=colours[colour_index], label=labels[i])
    colour_index += 1
plt.vlines(0, -90, 90, colors=['k'], linestyles=['--'])
plt.ylim(-90, 90)
plt.xlabel('Average difference (from the truth) ($\\degree$C)')
plt.ylabel('Latitude')
plt.legend(bbox_to_anchor=(1,1))